# `baseline_models.py` — Baseline Model Implementations

## Purpose
Implements the **comparison baselines** used to demonstrate that the full ClearView model (RoBERTa + GCN + Aspect Attention + Hybrid Loss) is meaningfully better than simpler alternatives.

## Baselines

| Class | Description | What it tests |
|-------|-------------|---------------|
| `PlainRoBERTa` | RoBERTa + single shared [CLS] head | Benefit of aspect-awareness |
| `CrossEntropyLossWrapper` | Same arch as full model + vanilla CE | Benefit of hybrid loss |
| `BERTBaseline` | BERT-base (not RoBERTa) | Encoder choice matters? |
| `TFIDFSVMBaseline` | TF-IDF features + LinearSVC per aspect | Deep learning vs. classical ML |

## Usage
Use the `create_baseline(name, config)` factory function — it's called by `experiment_runner.py` when it encounters a B-series experiment.

In [15]:
from tqdm.auto import tqdm
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
import torch
import torch.nn as nn
from transformers import RobertaModel, BertModel
import os, sys

try:
    ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    ROOT_DIR = os.path.dirname(os.path.abspath(""))

if ROOT_DIR not in sys.path: sys.path.append(ROOT_DIR)
if os.path.join(ROOT_DIR, "src") not in sys.path: sys.path.append(os.path.join(ROOT_DIR, "src"))
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
🕒 Done in 0.00s
✅ Completed: Loading dependencies and libraries.


## B1: Plain RoBERTa
The simplest transformer baseline. Uses [CLS] to represent the whole text and a single shared head for all aspects — aspect-unaware.

In [16]:
print("⏳ Starting: Defining class PlainRoBERTa...")
import time
_start_time = time.time()
class PlainRoBERTa(nn.Module):
    """
    RoBERTa + shared [CLS] head. No aspect identity passed into the model.
    Establishes the baseline benefit that just fine-tuning RoBERTa gives.
    """
    def __init__(self, roberta_model='roberta-base', num_classes=3, dropout=0.1):
        super(PlainRoBERTa, self).__init__()
        self.roberta    = RobertaModel.from_pretrained(roberta_model)  # Pre-trained encoder
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_classes)  # 768 = RoBERTa hidden dim

    def forward(self, input_ids, attention_mask, aspect_id=None, edge_index=None, **kwargs):
        # aspect_id is intentionally ignored — this model doesn't use aspect information
        output   = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls      = output.last_hidden_state[:, 0, :]  # Take [CLS] token representation
        cls      = self.dropout(cls)
        return self.classifier(cls)                   # Predict all 3 classes from [CLS]

    def get_num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class PlainRoBERTa.")


⏳ Starting: Defining class PlainRoBERTa...
🕒 Done in 0.00s
✅ Completed: Defining class PlainRoBERTa.


## B2: CrossEntropyLossWrapper
Not a different model — same architecture as the full model, but with standard Cross-Entropy loss instead of the Hybrid loss. This isolates the contribution of the custom loss functions.

In [17]:
print("⏳ Starting: Defining class CrossEntropyLossWrapper...")
import time
_start_time = time.time()
class CrossEntropyLossWrapper(nn.Module):
    """
    Wraps nn.CrossEntropyLoss with the same interface as AspectSpecificLossManager,
    so experiment_runner can call .compute_loss() identically for all experiments.
    """
    def __init__(self):
        super(CrossEntropyLossWrapper, self).__init__()
        self.criterion = nn.CrossEntropyLoss()  # Standard loss — no imbalance handling

    def compute_loss(self, predictions, targets, aspect_ids, aspect_names):
        """Identical signature to AspectSpecificLossManager.compute_loss() for compatibility."""
        loss     = self.criterion(predictions, targets)
        loss_val = loss.item()
        return loss, {'ce': loss_val, 'total': loss_val}  # match expected dict shape
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class CrossEntropyLossWrapper.")


⏳ Starting: Defining class CrossEntropyLossWrapper...
🕒 Done in 0.00s
✅ Completed: Defining class CrossEntropyLossWrapper.


## B3: BERT Baseline

In [18]:
print("⏳ Starting: Defining class BERTBaseline...")
import time
_start_time = time.time()
class BERTBaseline(nn.Module):
    """
    BERT-base-uncased with a single shared [CLS] classification head.
    Same as PlainRoBERTa but with BERT encoder — tests whether RoBERTa's extra
    pre-training (dynamic masking, larger corpus) gives a meaningful boost.
    """
    def __init__(self, bert_model='bert-base-uncased', num_classes=3, dropout=0.1):
        super(BERTBaseline, self).__init__()
        self.bert       = BertModel.from_pretrained(bert_model)  # BERT (not RoBERTa)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask, aspect_id=None, edge_index=None, **kwargs):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls    = output.last_hidden_state[:, 0, :]
        cls    = self.dropout(cls)
        return self.classifier(cls)

    def get_num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class BERTBaseline.")


⏳ Starting: Defining class BERTBaseline...
🕒 Done in 0.00s
✅ Completed: Defining class BERTBaseline.


## B4: TF-IDF + LinearSVC
A classical machine learning baseline. One TF-IDF vectoriser + SVM pipeline per aspect. No GPU, no transformers — shows how much deep learning adds beyond traditional bag-of-words NLP.

In [19]:
print("⏳ Starting: Defining class TFIDFSVMBaseline...")
import time
_start_time = time.time()
class TFIDFSVMBaseline:
    """
    Classical TF-IDF + LinearSVC baseline.
    One 3-class (neg/neu/pos) SVM pipeline per aspect.
    Uses sklearn — no GPU required.
    """
    def __init__(self, aspect_names, max_features=50000, ngram_range=(1, 2)):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.svm import LinearSVC
        from sklearn.calibration import CalibratedClassifierCV  # Adds predict_proba to LinearSVC
        from sklearn.pipeline import Pipeline

        self.aspect_names = aspect_names
        self.pipelines    = {}  # One trained pipeline per aspect

        for aspect in aspect_names:
            self.pipelines[aspect] = Pipeline([
                ('tfidf', TfidfVectorizer(
                    max_features=max_features,   # Limit vocabulary size
                    ngram_range=ngram_range,      # Unigrams + bigrams
                    sublinear_tf=True,            # Apply log scaling to TF values
                    strip_accents='unicode',
                    analyzer='word',
                    min_df=2,                     # Ignore tokens appearing < 2 times
                )),
                ('clf', CalibratedClassifierCV(LinearSVC(
                    class_weight='balanced',      # Handles class imbalance at training
                    max_iter=2000,
                    C=1.0,                        # Regularisation strength
                ), cv=3)),                        # cv=3 works with small per-class counts
            ])

    def fit(self, df, label_map):
        """Train one SVM pipeline per aspect on rows where that aspect is labelled."""
        for aspect in self.aspect_names:
            mask = df[aspect].notna()          # Only use rows that have a label for this aspect
            if mask.sum() == 0: continue
            X = df.loc[mask, 'data'].astype(str).tolist()
            y = df.loc[mask, aspect].map(lambda v: label_map.get(str(v).lower(), -1)).tolist()
            valid_pairs = [(x, lbl) for x, lbl in zip(X, y) if lbl != -1]
            if not valid_pairs: continue
            X_v, y_v = zip(*valid_pairs)
            print(f'  Training SVM for {aspect}: {len(X_v)} samples')
            self.pipelines[aspect].fit(X_v, y_v)

    def predict(self, texts, aspect):
        """Predict class labels (integers) for a list of texts for the given aspect."""
        return self.pipelines[aspect].predict(texts)

    def predict_proba(self, texts, aspect):
        """Returns probability estimates [neg, neu, pos] for the given aspect."""
        return self.pipelines[aspect].predict_proba(texts)
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class TFIDFSVMBaseline.")


⏳ Starting: Defining class TFIDFSVMBaseline...
🕒 Done in 0.00s
✅ Completed: Defining class TFIDFSVMBaseline.


## Factory Function

In [20]:
print("⏳ Starting: Defining function create_baseline...")
import time
_start_time = time.time()
def create_baseline(baseline_name: str, config: dict):
    """
    Factory: returns the right baseline model for a given name.
    Called by experiment_runner when encountering B-series experiments.
    """
    num_classes = config['model']['num_classes']   # Always 3 (neg/neu/pos)
    dropout     = config['model']['dropout']
    aspects     = config['aspects']['names']

    if baseline_name == 'plain_roberta':
        return PlainRoBERTa(config['model']['roberta_model'], num_classes, dropout)

    elif baseline_name == 'ce_loss':
        from models.model import create_model
        print("🧩 Building the neural network architecture...")
        return create_model(config)   # Full architecture, loss is replaced by CrossEntropyLossWrapper

    elif baseline_name == 'bert_base':
        return BERTBaseline('bert-base-uncased', num_classes, dropout)

    elif baseline_name == 'tfidf_svm':
        return TFIDFSVMBaseline(aspect_names=aspects)

    else:
        raise ValueError(f"Unknown baseline: '{baseline_name}'")
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function create_baseline.")


⏳ Starting: Defining function create_baseline...
🕒 Done in 0.00s
✅ Completed: Defining function create_baseline.


## Quick Test

Tests `PlainRoBERTa` with a dummy forward pass (no checkpoint needed — uses randomly initialised weights) and `TFIDFSVMBaseline` with a small in-memory dataset.

In [21]:
import time, pandas as pd
_start = time.time()
print("Running baseline_models quick test...\n")

BATCH = 4
SEQ   = 32
CLASSES = 3

# --- B1: PlainRoBERTa forward pass ---
print("Testing PlainRoBERTa...")
model = PlainRoBERTa(num_classes=CLASSES)
model.eval()
input_ids      = torch.randint(0, 50265, (BATCH, SEQ))
attention_mask = torch.ones(BATCH, SEQ, dtype=torch.long)
with torch.no_grad():
    logits = model(input_ids, attention_mask)
assert logits.shape == (BATCH, CLASSES), f"Expected ({BATCH},{CLASSES}), got {logits.shape}"
preds = logits.argmax(dim=1).tolist()
labels = ['negative','neutral','positive']
print(f"  Input:   {BATCH} samples × {SEQ} tokens")
print(f"  Output:  logits {logits.shape}")
print(f"  Preds:   {[labels[p] for p in preds]}")
print(f"  Params:  {model.get_num_parameters():,}")
print(f"  [OK]")

# --- B2: CrossEntropyLossWrapper ---
print("\nTesting CrossEntropyLossWrapper...")
ce_wrapper = CrossEntropyLossWrapper()
aspect_names = ['colour','smell','price','stayingpower']
targets      = torch.tensor([2, 0, 1, 2])
aspect_ids   = torch.tensor([0, 1, 2, 3])
loss, info   = ce_wrapper.compute_loss(logits, targets, aspect_ids, aspect_names)
print(f"  CE loss: {loss.item():.4f}")
print(f"  Info:    {info}")
print(f"  [OK]")

# --- B4: TFIDFSVMBaseline with tiny in-memory data ---
print("\nTesting TFIDFSVMBaseline...")
aspect_names_all = ['colour', 'smell', 'price']
label_map        = {'negative': 0, 'neutral': 1, 'positive': 2}

# Small dataset — enough to train a basic SVM
train_rows = [
    {'data': 'The colour is amazing and vibrant',      'colour': 'positive', 'smell': float('nan'), 'price': float('nan')},
    {'data': 'Colour is terrible and muddy',           'colour': 'negative', 'smell': float('nan'), 'price': float('nan')},
    {'data': 'Colour is ok nothing special',           'colour': 'neutral',  'smell': float('nan'), 'price': float('nan')},
    {'data': 'The smell is really awful',              'colour': float('nan'), 'smell': 'negative', 'price': float('nan')},
    {'data': 'Love the scent so much',                 'colour': float('nan'), 'smell': 'positive', 'price': float('nan')},
    {'data': 'Neutral smell, nothing strong',          'colour': float('nan'), 'smell': 'neutral',  'price': float('nan')},
    {'data': 'Way too expensive for what you get',     'colour': float('nan'), 'smell': float('nan'), 'price': 'negative'},
    {'data': 'Great value, very affordable',           'colour': float('nan'), 'smell': float('nan'), 'price': 'positive'},
    {'data': 'Price is average, not cheap not dear',   'colour': float('nan'), 'smell': float('nan'), 'price': 'neutral'},
] * 4  # repeat to give SVM enough samples

df_train = pd.DataFrame(train_rows)
svm      = TFIDFSVMBaseline(aspect_names_all)
svm.fit(df_train, label_map)

test_texts = ['The colour looks beautiful', 'The smell is strong and nice', 'Reasonable price']
for text, aspect in zip(test_texts, aspect_names_all):
    pred  = svm.predict([text], aspect)[0]
    proba = svm.predict_proba([text], aspect)[0]
    print(f"  [{aspect}] '{text}'")
    print(f"    pred={labels[pred]}  proba={[f'{p:.2f}' for p in proba]}")
print(f"  [OK]")

print(f"\nAll baseline_models tests PASSED  ({time.time() - _start:.2f}s)")

Running baseline_models quick test...

Testing PlainRoBERTa...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Input:   4 samples × 32 tokens
  Output:  logits torch.Size([4, 3])
  Preds:   ['positive', 'positive', 'positive', 'positive']
  Params:  124,647,939
  [OK]

Testing CrossEntropyLossWrapper...
  CE loss: 1.0894
  Info:    {'ce': 1.0894008874893188, 'total': 1.0894008874893188}
  [OK]

Testing TFIDFSVMBaseline...
  Training SVM for colour: 12 samples
  Training SVM for smell: 12 samples
  Training SVM for price: 12 samples
  [colour] 'The colour looks beautiful'
    pred=positive  proba=['0.25', '0.26', '0.49']
  [smell] 'The smell is strong and nice'
    pred=negative  proba=['0.48', '0.30', '0.23']
  [price] 'Reasonable price'
    pred=neutral  proba=['0.29', '0.41', '0.29']
  [OK]

All baseline_models tests PASSED  (1.36s)
